***IMPORTS , importing used libraries***

In [1]:
%pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-core faiss-cpu fastapi uvicorn pyngrok nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.

In [2]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.runnables import RunnablePassthrough,RunnableLambda
import re
from langchain_huggingface import ChatHuggingFace

/tmp/ipykernel_58/1885381513.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [3]:
#helper functions

def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"


def add_line_numbers(code: str) -> str:
    return "\n".join(
        f"{i+1:4} | {line}"
        for i, line in enumerate(code.splitlines())
    )

def review_python_file(file_path: str):
    # Read the file
    with open(file_path, "r", encoding="utf-8") as f:
        code = f.read()

    # Add line numbers
    numbered_code = add_line_numbers(code)

    # Run the AI review
    response = chain.invoke(numbered_code)

    return response

def safe_extract(text):
    try:
        block = extract_json_block(text)
        return block.strip("`").replace("json\n", "", 1)
    except Exception:
        return text

***loading files from cheatsheets directory***

In [4]:
cheatsheets_path = "/kaggle/input/datasets/yassinharraz/cheatsheet"
loader = DirectoryLoader(
    cheatsheets_path,
    glob="**/*.md",
    loader_cls=TextLoader
)

documents = loader.load()

In [5]:
len(documents)

121

In [6]:
#documents[77]

***now we will choose and implement chunking size and method***

In [7]:
#creating tet splitter to split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150
)
chunks = text_splitter.split_documents(documents)

In [8]:
len(chunks)

6125

In [9]:
print(chunks[0].page_content)

# Error Handling Cheat Sheet

## Introduction

Error handling is a part of the overall security of an application. Except in movies, an attack always begins with a **Reconnaissance** phase in which the attacker will try to gather as much technical information (often *name* and *version* properties) as possible about the target, such as the application server, frameworks, libraries, etc.

Unhandled errors can assist an attacker in this initial phase, which is very important for the rest of the attack.


***now we will embedd chunks and store in   FAISS***

In [10]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={
        "normalize_embeddings": True
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
## storing embedding in faais vector database
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
type(vector_store)

langchain_community.vectorstores.faiss.FAISS

In [12]:
vector_store.index.ntotal

6125

**implementing retrieval part**

In [13]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
#verfiying it works
query = "How should Python variables be named?"
results = retriever.invoke(query)
len(results)

3

In [14]:
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print("=" * 60)
    print(doc.metadata["source"])
    print()
    print(doc.page_content)
    print("\n")

Result 1
/kaggle/input/datasets/yassinharraz/cheatsheet/cheatsheets/pep08.md

Function and Variable Names
Function names should be lowercase, with words separated by underscores as necessary to improve readability.

Variable names follow the same convention as function names.

mixedCase is allowed only in contexts where that’s already the prevailing style (e.g. threading.py), to retain backwards compatibility.

Function and Method Arguments
Always use self for the first argument to instance methods.

Always use cls for the first argument to class methods.


Result 2
/kaggle/input/datasets/yassinharraz/cheatsheet/cheatsheets/pep08.md

Global Variable Names
(Let’s hope that these variables are meant for use inside one module only.) The conventions are about the same as those for functions.

Modules that are designed for use via from M import * should use the __all__ mechanism to prevent exporting globals, or use the older convention of prefixing such globals with an underscore (which you

In [15]:
!pip install -U bitsandbytes>=0.46.1

In [16]:
#Qwen/Qwen2.5-7B-Instruct
from transformers import BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

# 1. Create the quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. Load the tokenizer and the quantized model
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=2048,
    temperature=0.2,
    do_sample=False,
    return_full_text=False,
)

llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=text_generation_pipeline))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [17]:
response = llm.invoke(
    "Explain what SQL Injection is in two sentences."
)

print(response)

Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


content='SQL injection is a code injection technique used to exploit security vulnerabilities in database management systems by executing malicious SQL statements that can manipulate or steal data. It occurs when user input is not properly sanitized before being included in SQL queries, allowing attackers to inject and execute their own SQL commands.' additional_kwargs={} response_metadata={} id='lc_run--019f95fc-cc1a-7041-8c0d-f66c68c54675-0' tool_calls=[] invalid_tool_calls=[]


In [18]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert Python code reviewer.

Your job is to review Python code using ONLY the provided documentation.

Review the code for:

- PEP8 violations
- Clean Code issues
- Security vulnerabilities
- Bugs
- Performance problems

For every issue you find:

- Explain why it is a problem.
- Suggest a fix.

If there are no issues, clearly state that no issues were found.

Return your response in a structured format.
"""
        ),
        (
            "human",
            """
Documentation:

{context}

----------------------------------------

Python Code:

{code}

----------------------------------------

Review this code.

Return your response using the following format:

{format_instructions}
"""
        ),
    ]
)

In [19]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [20]:
#output schema
from pydantic import BaseModel, Field

class Issue(BaseModel):
    severity: str = Field(
        description="Severity level: Low, Medium, or High"
    )

    line: int = Field(
        description="Line number where the issue occurs"
    )

    issue: str = Field(
        description="Short title of the detected issue"
    )

    explanation: str = Field(
        description="Detailed explanation of why this is a problem"
    )

    suggested_fix: str = Field(
        description="Recommended way to fix the issue"
    )


from typing import List

class CodeReviewReport(BaseModel):
    file: str = Field(
        description="Name of the analyzed Python file"
    )

    issues: List[Issue] = Field(
        description="List of detected issues"
    )

from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(
    pydantic_object=CodeReviewReport
) ##parser that  convert the LLM's output into a CodeReviewReport

In [21]:
chain = (
    {
        "context": retriever | format_docs,
        "code": RunnablePassthrough(),
        "format_instructions": RunnableLambda(
            lambda _: parser.get_format_instructions()
        ),
    }
    | prompt
    | llm
    | RunnableLambda(safe_extract)
    | parser
)

In [22]:
# sample_code = """
# def GetUser(ID):
#     query = "SELECT * FROM users WHERE id=" + ID
#     return query
# """

# response = chain.invoke(sample_code)

# print(response)
# #print(response.model_dump_json(indent=2))


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio
import uvicorn
from pyngrok import ngrok

# 1. Define the incoming request schema
class CodeRequest(BaseModel):
    code: str

# 2. Initialize the API
app = FastAPI()

# 3. Create the REST endpoint
@app.post("/review")
@app.post("/review")
def review_code_endpoint(request: CodeRequest):
    try:
        response = chain.invoke(request.code)
        return response
    except Exception as e:
        print("CHAIN FAILURE:", repr(e))   # <-- add this line
        return {"error": str(e)}

# 4. Set up the Ngrok Tunnel
# WARNING: Keep your auth token private!
NGROK_AUTH_TOKEN = "3GxM8ZaaWPaPV2XOP7ClnWI4l6F_4U8dvR6nqngQnWnaf7w7Q"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open a local tunnel on port 8000
ngrok_tunnel = ngrok.connect(8000)
public_url = ngrok_tunnel.public_url

print("=" * 60)
print(f"API is live!")
print(f"Public URL: {public_url}")
print(f"Send your POST requests to: {public_url}/review")
print("=" * 60)

# 5. Run the server inside the Kaggle notebook
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

# Await the server to attach it to Kaggle's existing event loop
await server.serve()

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


API is live!
Public URL: https://pox-basis-aids.ngrok-free.dev
Send your POST requests to: https://pox-basis-aids.ngrok-free.dev/review


Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     41.41.185.185:0 - "POST /review HTTP/1.1" 200 OK
